# Notebook 4: Sequential Target-Aware Attention Ranker
    
This notebook builds our advanced PyTorch architecture. Instead of global collaborative filtering, we treat this as a re-ranking task. For each candidate in the slate, the model learns an attention weight over the user's historical interactions, conditioned on the candidate item itself (Target-Aware Attention).


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
import time
import gc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


In [8]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import time
import gc

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

PROCESSED_DIR = Path(r"C:\Users\Acer\OneDrive - National University of Singapore\BT4222\JDsearch\processed")
cand_df = pd.read_parquet(PROCESSED_DIR / "candidates_split.parquet")
hist_df = pd.read_parquet(PROCESSED_DIR / "history_split.parquet")

print("Filtering to warm items (appear >= 3 times in train)...")
train_hist = hist_df[hist_df.split == "train"]
item_counts = train_hist["item_id"].value_counts()
warm_items = item_counts[item_counts >= 3].index

# Keep only warm items in both candidates and histories
cand_df = cand_df[cand_df["item_id"].isin(warm_items)].copy()
hist_df = hist_df[hist_df["item_id"].isin(warm_items)].copy()

# Map all unique item IDs to contiguous integers
all_items = pd.concat([cand_df["item_id"], hist_df["item_id"]]).unique()
item_to_idx = {item: i + 1 for i, item in enumerate(all_items)} # +1 to reserve 0 for padding
num_items = len(item_to_idx) + 1

cand_df["item_idx"] = cand_df["item_id"].map(item_to_idx).astype("int32")
hist_df["item_idx"] = hist_df["item_id"].map(item_to_idx).astype("int32")

print(f"Filtered Vocabulary size: {num_items:,}")

Using device: cpu
Filtering to warm items (appear >= 3 times in train)...
Filtered Vocabulary size: 1,139,017


## 1. Define the Slate Dataset

We process the data session-by-session. Each session has:
- A history sequence (padded to length 20).
- A candidate slate (padded to length 90).
- The ground truth label for each candidate (0=None, 1=Click, 2=Cart, 3=Purchase).


In [9]:
class SlateDataset(Dataset):
    def __init__(self, cand_df, hist_df):
        # Group history by session
        self.hist_map = hist_df.groupby("session_id")["item_idx"].apply(list).to_dict()
        
        # Group candidate slates by session
        self.slates = []
        for sid, group in cand_df.groupby("session_id"):
            self.slates.append({
                "sid": sid,
                "items": group["item_idx"].values,
                "labels": group["raw_action_label"].values
            })
            
    def __len__(self):
        return len(self.slates)
        
    def __getitem__(self, idx):
        slate = self.slates[idx]
        sid = slate["sid"]
        
        # 1. Pad History (Most recent 20 items)
        h = self.hist_map.get(sid, [])
        h = h[-20:] # Keep only the most recent 20
        h_padded = np.zeros(20, dtype=np.int64)
        # Safety guard: only try to assign if h has items
        if len(h) > 0:
            h_padded[-len(h):] = h
            
        # 2. Pad Slate (up to 90 items)
        i = slate["items"]
        i_padded = np.zeros(90, dtype=np.int64)
        i_padded[:len(i)] = i[:90]
        
        # 3. Pad Labels
        y_padded = np.zeros(90, dtype=np.float32)
        y_padded[:len(i)] = slate["labels"][:90]
        
        return torch.tensor(h_padded), torch.tensor(i_padded), torch.tensor(y_padded)

In [10]:
# Create datasets
train_ds = SlateDataset(cand_df[cand_df.split == "train"], hist_df[hist_df.split == "train"])
val_ds = SlateDataset(cand_df[cand_df.split == "val"], hist_df[hist_df.split == "val"])
test_ds = SlateDataset(cand_df[cand_df.split == "test"], hist_df[hist_df.split == "test"])

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")
gc.collect()

Train batches: 444 | Val batches: 95 | Test batches: 95


49

## 2. Define the Target-Aware Attention Model

For each candidate in the slate, we compute an attention score against every item in the user's history. The model then creates a context vector (a weighted sum of history items) and combines it with the candidate embedding to produce a final rank score.


In [11]:
class AttentionRanker(nn.Module):
    def __init__(self, num_items, embed_dim=32):
        super().__init__()
        self.item_emb = nn.Embedding(num_items, embed_dim, padding_idx=0)
        
        # Attention mechanism
        self.attn_dense = nn.Linear(embed_dim * 2, 1)
        
        # Final scoring MLP
        self.scorer = nn.Sequential(
            nn.Linear(embed_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
        
    def forward(self, hist, slate):
        # hist: [B, 20], slate: [B, 90]
        h_emb = self.item_emb(hist)    # [B, 20, D]
        s_emb = self.item_emb(slate)   # [B, 90, D]
        
        # Expand for pairwise attention
        h_exp = h_emb.unsqueeze(1).expand(-1, 90, -1, -1) # [B, 90, 20, D]
        s_exp = s_emb.unsqueeze(2).expand(-1, -1, 20, -1) # [B, 90, 20, D]
        
        # Concatenate and score
        cat = torch.cat([h_exp, s_exp], dim=-1)
        attn_scores = self.attn_dense(cat).squeeze(-1) # [B, 90, 20]
        
        # Mask zero-padding in history
        mask = (hist == 0).unsqueeze(1).expand(-1, 90, -1)
        attn_scores = attn_scores.masked_fill(mask, -1e9)
        
        # Softmax and context vector
        attn_weights = torch.softmax(attn_scores, dim=-1)
        # If history was entirely padding, softmax gives NaN. Fill with 0.
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)
        
        ctx = torch.sum(attn_weights.unsqueeze(-1) * h_exp, dim=2) # [B, 90, D]
        
        # Combine candidate and context
        combined = torch.cat([s_emb, ctx], dim=-1) # [B, 90, 2D]
        scores = self.scorer(combined).squeeze(-1) # [B, 90]
        return scores

model = AttentionRanker(num_items).to(DEVICE)
print(model)

AttentionRanker(
  (item_emb): Embedding(1139017, 32, padding_idx=0)
  (attn_dense): Linear(in_features=64, out_features=1, bias=True)
  (scorer): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


## 3. Training Loop (Listwise Softmax Loss)

Because the task is re-ranking a slate, we use a listwise loss. For each session, we normalize the labels into a probability distribution (e.g., purchasing an item gives a higher target probability than clicking). We then minimize the cross-entropy between the model's softmax scores and this target distribution.


In [12]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Ordinal gains: 0=None, 1=Click, 3=Cart, 7=Purchase
label_gains = torch.tensor([0.0, 1.0, 3.0, 7.0], device=DEVICE)

def listwise_loss(scores, labels, mask):
    # Convert ordinal labels to gains
    gains = label_gains[labels.long()]
    
    # Normalize gains into a target probability distribution
    target = gains / (gains.sum(dim=1, keepdim=True) + 1e-9)
    
    # Mask padded items in the slate
    scores = scores.masked_fill(mask == 0, -1e9)
    log_probs = torch.log_softmax(scores, dim=1)
    
    # Only compute loss for sessions that have at least one positive interaction
    has_positive = (gains.sum(dim=1) > 0).float()
    
    loss = -((target * log_probs).sum(dim=1) * has_positive).sum() / (has_positive.sum() + 1e-9)
    return loss

print("Starting training...")
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for hist, slate, labels in train_loader:
        hist, slate, labels = hist.to(DEVICE), slate.to(DEVICE), labels.to(DEVICE)
        
        # Create mask for valid slate items (non-padding)
        mask = (slate != 0).float()
        
        optimizer.zero_grad()
        scores = model(hist, slate)
        loss = listwise_loss(scores, labels, mask)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Time: {time.time() - start_time:.1f}s")

Starting training...


Epoch 1/3 | Loss: 3.0934 | Time: 541.8s
Epoch 2/3 | Loss: 3.0474 | Time: 537.4s
Epoch 3/3 | Loss: 2.9239 | Time: 350.4s


## 4. Evaluation (NDCG@10)

In [13]:
from sklearn.metrics import ndcg_score

model.eval()
ndcg_scores = []

print("Evaluating on Test Set...")
with torch.no_grad():
    for hist, slate, labels in test_loader:
        hist, slate, labels = hist.to(DEVICE), slate.to(DEVICE), labels.to(DEVICE)
        scores = model(hist, slate)
        
        # Move to CPU for sklearn metrics
        scores = scores.cpu().numpy()
        labels = labels.cpu().numpy()
        slate = slate.cpu().numpy()
        
        # Evaluate per session in the batch
        for i in range(scores.shape[0]):
            valid_mask = slate[i] != 0
            true_rel = labels[i][valid_mask]
            pred_rel = scores[i][valid_mask]
            
            if (true_rel > 0).sum() > 0 and len(true_rel) >= 2:
                ndcg = ndcg_score(true_rel.reshape(1, -1), pred_rel.reshape(1, -1), k=10)
                ndcg_scores.append(ndcg)

print(f"Mean NDCG@10 on Test Set: {np.mean(ndcg_scores):.4f}")
print(f"Evaluated on {len(ndcg_scores)} sessions.")

Evaluating on Test Set...
Mean NDCG@10 on Test Set: 0.3384
Evaluated on 18007 sessions.
